# Backpressure and Scalability

## Overview

This notebook studies what happens when profiling events are generated faster than the downstream pipeline can process, transmit, or store them.

The previous notebooks established the basic profiling data path:

```text
ProfileEvent
    ↓
BoundedQueue
    ↓
Serialization
    ↓
Framing
    ↓
Batching
    ↓
TCP
    ↓
Collector
```

So far, most experiments assumed that the downstream stages were fast enough to keep up with event production.

In a real profiling system, this assumption may fail.

For example:

```text
Producer rate:
100,000 events / second

Consumer rate:
30,000 events / second
```

If the system uses an unbounded queue, events continue accumulating:

```text
Producer
   ↓
Queue
[events]
[more events]
[more events]
[more events]
   ↓
Consumer
```

This can cause:

```text
increasing memory usage
high queueing latency
event backlog
eventual out-of-memory failure
```

To prevent this, the profiling pipeline needs a controlled overload strategy.

---

## Core Question

The central question of this notebook is:

> What should a profiling system do when events are generated faster than they can be consumed?

We will study several possible answers:

```text
Block the producer

Drop the newest event

Drop the oldest event

Increase batching efficiency
```

Each approach has different tradeoffs.

---

## Backpressure

Backpressure occurs when a downstream component cannot process data as quickly as an upstream component produces it.

For example:

```text
Fast Producer
      ↓
Bounded Queue
████████████ FULL
      ↓
Slow Consumer
```

Once the queue becomes full, the system must react.

With a blocking queue:

```text
queue full
    ↓
producer waits
    ↓
consumer removes an event
    ↓
space becomes available
    ↓
producer continues
```

The slowdown therefore propagates backward through the pipeline.

This is called:

```text
backpressure propagation
```

A larger system may look like:

```text
Slow Collector
      ↓
TCP Sender slows down
      ↓
Consumer slows down
      ↓
Queue fills
      ↓
Producer blocks or drops events
```

---

## Why Backpressure Matters for Profiling

A profiler should observe an application while disturbing it as little as possible.

If producing a profiling event can block the application thread:

```text
Application
    ↓
record profiling event
    ↓
queue full
    ↓
wait
```

then the profiler itself can change the application's execution time.

For example:

```text
Without profiler:
100 ms

With heavy profiler backpressure:
150 ms
```

This introduces:

```text
profiling overhead
```

and can make performance measurements less representative of the original application.

Therefore, profiling infrastructure must balance:

```text
data completeness
        vs
profiling overhead
        vs
memory usage
        vs
latency
```

---

# Notebook Plan

This notebook is divided into eight sections.

---

## 1. Standalone Setup and Metrics Design

We will recreate only the components required for the experiments:

```text
ProfileEvent
BoundedQueue
producer thread
consumer thread
timing utilities
```

We will also define the metrics used throughout the notebook.

Examples include:

```text
events produced
events consumed
events dropped

producer throughput
consumer throughput

queue occupancy
maximum queue occupancy

producer blocked time

total runtime
```

The goal is to make backpressure measurable rather than only observable.

---

## 2. Fast Producer and Slow Consumer Baseline

We will intentionally create an overloaded pipeline.

For example:

```text
Producer:
very fast

Queue:
capacity = 8

Consumer:
10 ms processing delay per event
```

The expected behavior is:

```text
Producer produces quickly
        ↓
Queue fills
        ↓
Queue becomes full
        ↓
Producer blocks
        ↓
Consumer slowly releases space
```

This experiment provides the baseline backpressure behavior.

---

## 3. Measure Queue Occupancy and Producer Blocking

The basic experiment will be extended with instrumentation.

We want to measure:

```text
maximum queue occupancy

number of times producer encounters backpressure

total producer blocked time

average blocked time

consumer throughput
```

For example:

```text
Experiment runtime:
10 seconds

Producer blocked:
7.8 seconds
```

This tells us that the producer spends most of its time waiting for downstream processing.

---

## 4. Queue Capacity Experiment

We will compare several queue capacities:

```text
1
8
64
256
1024
```

The purpose is to study the relationship between:

```text
queue capacity
    ↓
memory usage
    ↓
producer blocking
    ↓
queueing latency
```

A very small queue may cause frequent blocking:

```text
small queue
    ↓
low memory usage
    ↓
frequent producer stalls
```

A larger queue may reduce short-term blocking:

```text
large queue
    ↓
more buffering
    ↓
less immediate producer blocking
```

but may increase:

```text
memory usage
event queueing latency
```

Therefore:

```text
larger queue != always better
```

---

## 5. Compare Overload Policies

We will compare three queue-overload strategies.

### Block Producer

```text
Queue full
    ↓
Producer waits
```

Advantages:

```text
no event loss
complete profiling data
```

Disadvantages:

```text
producer may stall
application interference may increase
```

---

### Drop Newest

```text
Queue:

[A][B][C][D]

New event:
E

Queue full
    ↓
drop E
```

Advantages:

```text
producer does not block
lower application interference
```

Disadvantages:

```text
profiling events are lost
```

---

### Drop Oldest

```text
Queue:

[A][B][C][D]

New event:
E

drop A
    ↓
[B][C][D][E]
```

This keeps the newest information at the cost of older events.

We will compare:

```text
throughput
drop rate
producer blocking
runtime
```

---

## 6. Add Batching to the Consumer

The consumer will then process several events together.

Instead of:

```text
pop event
    ↓
process event
```

we move toward:

```text
pop multiple events
        ↓
build batch
        ↓
process/send batch
```

For example:

```text
BoundedQueue
     ↓
pop 16 events
     ↓
BatchBuilder
     ↓
one downstream operation
```

We will compare different batch sizes:

```text
1
4
16
64
```

The expected tradeoff is:

```text
larger batch
    ↓
fewer downstream operations
    ↓
better throughput
```

but potentially:

```text
larger batch
    ↓
events wait longer
    ↓
higher latency
```

This introduces the classic:

```text
throughput vs latency
```

tradeoff.

---

## 7. Simulate Slow Network or Collector

We will artificially slow the downstream stage to simulate:

```text
network congestion
slow TCP transmission
slow collector
slow database writes
slow analysis pipeline
```

For example, the consumer may deliberately sleep after processing a batch.

This allows us to observe:

```text
Slow downstream
      ↓
Consumer throughput decreases
      ↓
Queue occupancy rises
      ↓
Backpressure reaches producer
```

This experiment demonstrates backpressure propagation across the pipeline.

---

## 8. Final Scalability Benchmark

Finally, we will compare several configurations.

Parameters may include:

```text
queue capacity

batch size

consumer delay

overload policy
```

The results can be summarized in a table such as:

| Policy | Queue Capacity | Batch Size | Throughput | Drop Rate | Producer Block Time |
|---|---:|---:|---:|---:|---:|
| Block | 8 | 1 | ... | 0% | ... |
| Block | 64 | 16 | ... | 0% | ... |
| Drop Newest | 64 | 16 | ... | ... | 0 |
| Drop Oldest | 64 | 16 | ... | ... | 0 |

The goal is not to find one universally correct configuration.

The goal is to understand the tradeoffs and explain why one configuration may be appropriate for a particular profiling workload.

---

# Overall Experimental Flow

```text
Fast Producer
      ↓
BoundedQueue
      ↓
Slow Consumer
      ↓
Observe backpressure
      ↓
Measure blocking and occupancy
      ↓
Change queue capacity
      ↓
Compare overload policies
      ↓
Add batching
      ↓
Simulate slow downstream
      ↓
Run scalability benchmark
      ↓
Analyze system tradeoffs
```

---

# Key Design Tradeoffs

Throughout the notebook, we will repeatedly evaluate four competing objectives:

```text
Data completeness
        ↕
Application interference

Memory usage
        ↕
Queueing capacity

Throughput
        ↕
Latency

Backpressure
        ↕
Event dropping
```

A profiling system must choose its behavior based on the importance of each objective.

---

# Interview-Relevant Questions

By the end of this notebook, the following questions should be answerable:

- What is backpressure?
- Why is an unbounded queue dangerous?
- How does a bounded queue control memory growth?
- What happens when the producer is faster than the consumer?
- How can producer blocking affect profiling accuracy?
- How would you measure backpressure?
- How does queue capacity affect blocking and latency?
- What are the tradeoffs between blocking and dropping events?
- What is the difference between drop-newest and drop-oldest policies?
- How can batching improve throughput?
- Why can larger batches increase latency?
- How does downstream slowdown propagate upstream?
- How would you design an overload policy for a production profiling system?

---

# Final Goal

By the end of this notebook, the profiling pipeline should evolve from:

```text
Producer
    ↓
Queue
    ↓
Consumer
```

into a measurable and configurable system:

```text
                    metrics
                       ↑
                       │
Producer
    ↓
Overload Policy
    ↓
BoundedQueue
    ↓
Batching Consumer
    ↓
Simulated Network / Collector
    ↓
Performance Analysis
```

The final objective is to be able to explain not only how the pipeline works, but also how it behaves under overload and why particular scalability decisions were made.

## Standalone Setup + Metrics Design

### Goal

The goal of this section is to build a small, self-contained environment for studying backpressure.

We do **not** add TCP, serialization, framing, or storage yet.

We first isolate the producer-consumer system:

```text
Producer
    ↓
BoundedQueue
    ↓
Consumer
```

and make its behavior measurable.

The main question is:

> When the producer is faster than the consumer, how much backpressure is created and how can we quantify it?

---

## Components Used

For the first experiment, we only need:

```text
ProfileEvent
BoundedQueue<ProfileEvent>
producer thread
consumer thread
timing utilities
metrics counters
```

We intentionally keep the system small so that any performance effect can be attributed mainly to:

```text
producer rate
consumer rate
queue capacity
```

rather than to networking or serialization overhead.

---

## Experimental Architecture

The initial experiment will use:

```text
Fast Producer
      ↓
BoundedQueue
      ↓
Slow Consumer
```

The producer generates events as quickly as possible.

The consumer intentionally processes them more slowly.

For example:

```text
Producer:
no artificial delay

Consumer:
sleep 5 ms per event
```

with:

```text
queue capacity = 8
```

This should cause the queue to fill quickly.

---

## Expected Backpressure Behavior

At the beginning:

```text
Producer
    ↓
[ ][ ][ ][ ][ ][ ][ ][ ]
    ↓
Consumer
```

The producer starts filling the queue:

```text
Producer
    ↓
[A][B][C][D][E][F][ ][ ]
    ↓
Consumer
```

Eventually:

```text
Producer
    ↓
[A][B][C][D][E][F][G][H]
                         ↑
                        FULL
    ↓
Consumer
```

Once the queue is full:

```text
producer push()
      ↓
wait
      ↓
consumer removes one event
      ↓
space becomes available
      ↓
producer continues
```

This is the simplest form of backpressure.

---

# Metrics Design

To understand the behavior of the system, we need more than just program output.

We need quantitative metrics.

The main metrics used in this notebook will be:

```text
events produced
events consumed
events dropped
queue occupancy
maximum queue occupancy
producer blocking count
producer blocked time
producer throughput
consumer throughput
total runtime
```

---

## Events Produced

This metric counts how many events the producer successfully generates.

Conceptually:

```text
producer creates event
        ↓
produced_events++
```

For the blocking policy, this will usually equal the number of events eventually inserted into the queue.

Example:

```text
Produced events: 1000
```

---

## Events Consumed

This counts how many events are successfully processed by the consumer.

Conceptually:

```text
consumer pops event
        ↓
process event
        ↓
consumed_events++
```

For a system that does not drop events, we expect:

```text
produced_events == consumed_events
```

after the pipeline finishes.

---

## Events Dropped

Later, when we introduce overload policies such as:

```text
drop newest
drop oldest
```

we will count how many events are discarded.

For the first blocking experiment:

```text
dropped_events = 0
```

because the producer waits instead of dropping data.

---

## Queue Occupancy

Queue occupancy means:

```text
number of events currently stored in the queue
```

For example:

```text
capacity = 8

current queue:

[A][B][C][D][ ][ ][ ][ ]

occupancy = 4
```

Queue occupancy gives us a direct view of backpressure.

If the queue frequently stays near capacity:

```text
occupancy ≈ capacity
```

then the consumer is not keeping up with the producer.

---

## Maximum Queue Occupancy

We will also record:

```text
max_queue_occupancy
```

Example:

```text
capacity = 64

maximum observed occupancy = 64
```

This means the queue became completely full at least once.

If:

```text
maximum observed occupancy = 7
```

then a capacity of 64 may be much larger than necessary for that workload.

---

## Producer Blocking Count

For the blocking queue policy, we want to know how often the producer encounters a full queue.

Conceptually:

```text
producer wants to push
        ↓
queue full?
        ↓
yes
        ↓
producer blocks
```

Each such occurrence can increment:

```text
producer_block_count
```

This tells us how frequently backpressure reaches the producer.

---

## Producer Blocked Time

This is one of the most important metrics.

For every `push()` operation, we can measure:

```text
start time
    ↓
queue.push(...)
    ↓
end time
```

If the queue has free space, `push()` returns quickly.

If the queue is full, `push()` may wait.

Therefore:

```text
push duration
≈
queue waiting time
+
small insertion overhead
```

The total blocked time can be accumulated across all push operations.

For example:

```text
Total runtime:
5.0 seconds

Producer blocked time:
4.1 seconds
```

This indicates very strong backpressure.

---

## Producer Throughput

Producer throughput can be defined as:

```text
produced events
----------------
total runtime
```

For example:

```text
10,000 events
--------------
2 seconds

= 5,000 events/second
```

This tells us the effective event-generation rate after accounting for blocking.

---

## Consumer Throughput

Consumer throughput is:

```text
consumed events
----------------
total runtime
```

For a stable blocking pipeline, producer and consumer throughput should eventually become similar because the slower stage controls the overall system throughput.

For example:

```text
Producer capability:
100,000 events/s

Consumer capability:
200 events/s
```

With a bounded blocking queue, long-term pipeline throughput will be much closer to:

```text
200 events/s
```

because the consumer is the bottleneck.

---

## Total Runtime

We will measure the complete experiment using:

```cpp
std::chrono::steady_clock
```

Conceptually:

```text
experiment start
      ↓
producer + consumer run
      ↓
threads finish
      ↓
experiment end
```

Then:

```text
runtime = end - start
```

`steady_clock` is appropriate for performance measurement because it is monotonic and is not affected by wall-clock adjustments.

---

# Initial Metrics Structure

Conceptually, the experiment will maintain something like:

```text
Metrics
│
├── produced_events
├── consumed_events
├── dropped_events
├── max_queue_occupancy
├── producer_block_count
├── producer_blocked_time
└── total_runtime
```

From these raw values, we can calculate:

```text
producer throughput

consumer throughput

drop rate

percentage of runtime blocked
```

For example:

```text
blocked_percentage =
producer_blocked_time
----------------------
total_runtime
× 100%
```

---

# Why These Metrics Matter

Suppose two queue configurations produce the same throughput:

```text
Queue capacity = 8

throughput = 1000 events/s
producer blocked = 80% of runtime
```

and:

```text
Queue capacity = 64

throughput = 1000 events/s
producer blocked = 30% of runtime
```

The throughput alone does not tell the whole story.

The second configuration may create less immediate pressure on the producer.

However, it may also create:

```text
more buffered events
more memory usage
higher queueing latency
```

Therefore, scalability cannot be evaluated using only one metric.

---

# First Experiment Configuration

For the baseline experiment, we will start with a deliberately overloaded configuration:

```text
Number of events:
about 100–1000

Queue capacity:
8

Producer delay:
0 ms

Consumer delay:
5–10 ms per event
```

The exact values can be adjusted after the first run.

We expect:

```text
queue fills quickly

producer frequently blocks

consumer becomes the bottleneck

no events are dropped
```

---

# Expected Result

Conceptually:

```text
Producer starts
      ↓
produces very quickly
      ↓
queue reaches capacity
      ↓
producer blocks
      ↓
consumer processes one event
      ↓
producer inserts another event
      ↓
queue stays close to full
      ↓
repeat
```

The metrics should therefore show something like:

```text
Produced events:        100
Consumed events:        100
Dropped events:           0

Max queue occupancy:      8

Producer block count:    high
Producer blocked time:   high

Producer throughput:     close to consumer throughput
Consumer throughput:     limited by artificial delay
```

---

# What We Are NOT Testing Yet

This section intentionally does not include:

```text
serialization
framing
batching
TCP
database storage
GPU profiling
```

Those components would make the experiment harder to interpret.

The goal here is to isolate:

```text
producer
    ↓
queue
    ↓
consumer
```

and understand backpressure behavior clearly before adding more system components.

---

# Questions to Answer After This Section

By the end of this section, the following questions should be answerable:

- What metrics are useful for measuring backpressure?
- Why is queue occupancy important?
- What does maximum queue occupancy tell us?
- How can producer blocked time be measured?
- Why can producer throughput become limited by consumer throughput?
- Why is `std::chrono::steady_clock` appropriate for performance measurement?
- Why is throughput alone insufficient to evaluate a queue design?
- Why should networking and serialization be excluded from the first backpressure experiment?
- What result would indicate that the consumer is the bottleneck?

In [1]:
!apt-get update -qq
!apt-get install -y -qq build-essential g++ cmake ninja-build git

!mkdir -p distributed-profiler/include
!mkdir -p distributed-profiler/src
!mkdir -p distributed-profiler/tests
!mkdir -p distributed-profiler/benchmarks
!mkdir -p distributed-profiler/apps

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ninja-build.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.11.1-2_amd64.deb ...
Unpacking ninja-build (1.11.1-2) ...
Setting up ninja-build (1.11.1-2) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
%%writefile distributed-profiler/include/profile_event.hpp

#pragma once

#include <cstdint>
#include <string>
#include <variant>

struct KernelEvent {
    std::uint64_t timestamp_ns;
    std::uint64_t duration_ns;
    std::uint32_t device_id;
    std::string kernel_name;
};

struct MemoryEvent {
    std::uint64_t timestamp_ns;
    std::uint32_t device_id;
    std::uint64_t bytes;
    std::string operation;
};

using ProfileEvent =
    std::variant<KernelEvent, MemoryEvent>;

Writing distributed-profiler/include/profile_event.hpp


In [3]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <stop_token>
#include <utility>

template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity)
        : capacity_(capacity),
          closed_(false) {}

    bool push(
        T item,
        std::stop_token stop_token
    ) {
        std::unique_lock<std::mutex> lock(mutex_);

        bool ready = not_full_.wait(
            lock,
            stop_token,
            [this]() {
                return queue_.size() < capacity_
                    || closed_;
            }
        );

        if (
            !ready ||
            stop_token.stop_requested() ||
            closed_
        ) {
            return false;
        }

        queue_.push_back(std::move(item));

        lock.unlock();
        not_empty_.notify_one();

        return true;
    }


    std::optional<T> pop(
        std::stop_token stop_token
    ) {
        std::unique_lock<std::mutex> lock(mutex_);

        bool ready = not_empty_.wait(
            lock,
            stop_token,
            [this]() {
                return !queue_.empty()
                    || closed_;
            }
        );

        if (
            !ready ||
            stop_token.stop_requested()
        ) {
            return std::nullopt;
        }

        if (
            queue_.empty() &&
            closed_
        ) {
            return std::nullopt;
        }

        T item =
            std::move(queue_.front());

        queue_.pop_front();

        lock.unlock();
        not_full_.notify_one();

        return item;
    }


    void close() {
        {
            std::lock_guard<std::mutex> lock(mutex_);
            closed_ = true;
        }

        not_empty_.notify_all();
        not_full_.notify_all();
    }


private:
    std::deque<T> queue_;
    std::size_t capacity_;

    std::mutex mutex_;

    std::condition_variable_any not_empty_;
    std::condition_variable_any not_full_;

    bool closed_;
};

Writing distributed-profiler/include/bounded_queue.hpp


In [4]:
%%writefile distributed-profiler/include/pipeline_metrics.hpp

#pragma once

#include <chrono>
#include <cstddef>
#include <cstdint>

struct PipelineMetrics {

    // ============================================================
    // Workload metrics
    // ============================================================

    // Number of events the producer attempted to generate.
    std::uint64_t attempted_events = 0;

    // Number of events successfully admitted into the pipeline.
    std::uint64_t produced_events = 0;

    // Number of events successfully consumed.
    std::uint64_t consumed_events = 0;

    // Number of events dropped.
    //
    // For the initial blocking policy, this should stay at 0.
    std::uint64_t dropped_events = 0;


    // ============================================================
    // Backpressure metrics
    // ============================================================

    // Number of times the producer encounters backpressure.
    std::uint64_t producer_block_count = 0;

    // Total time spent waiting while trying to push events.
    std::chrono::nanoseconds producer_blocked_time{0};


    // ============================================================
    // Queue metrics
    // ============================================================

    // Maximum number of events observed in the queue.
    std::size_t max_queue_occupancy = 0;


    // ============================================================
    // Experiment timing
    // ============================================================

    // Total runtime of the experiment.
    std::chrono::nanoseconds total_runtime{0};
};

Writing distributed-profiler/include/pipeline_metrics.hpp


In [5]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <stop_token>
#include <utility>

template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity)
        : capacity_(capacity),
          closed_(false) {}


    // ============================================================
    // Push one item into the queue.
    //
    // If the queue is full, wait until:
    //
    // - space becomes available
    // - queue is closed
    // - stop is requested
    // ============================================================

    bool push(
        T item,
        std::stop_token stop_token
    ) {
        // Create a unique_lock for mutex_.
        std::unique_lock<std::mutex> lock(mutex_);

        // Wait on not_full_.
        bool ready = not_full_.wait(
            lock,
            stop_token,
            [this]() {
                return queue_.size() < capacity_
                    || closed_;
            }
        );


        if (
            !ready ||
            stop_token.stop_requested() ||
            closed_
        ) {
            return false;
        }


        // Move item into queue_.
        queue_.push_back(std::move(item));

        // Unlock mutex before notification.
        lock.unlock();

        // Notify one waiting consumer.
        not_empty_.notify_one();

        // Return true.
        return true;
    }


    // ============================================================
    // Pop one item from the queue.
    //
    // If the queue is empty, wait until:
    //
    // - an item becomes available
    // - queue is closed
    // - stop is requested
    // ============================================================

    std::optional<T> pop(
        std::stop_token stop_token
    ) {
        // Create a unique_lock for mutex_.
        std::unique_lock<std::mutex> lock(mutex_);

        bool ready = not_empty_.wait(
            lock,
            stop_token,
            [this]() {
                return !queue_.empty()
                    || closed_;
            }
        );


        // If wait is interrupted or stop requested:
        //
        // return std::nullopt.
        if (
            !ready ||
            stop_token.stop_requested()
        ) {
            return std::nullopt;
        }

        // If queue is empty AND closed:
        //
        // return std::nullopt.
        if (
            queue_.empty() &&
            closed_
        ) {
            return std::nullopt;
        }


        // Move the front item into a local variable.
        T item = std::move(queue_.front());


        // Remove the front item from queue_.
        queue_.pop_front();

        // Unlock mutex.
        lock.unlock();


        // Notify one waiting producer.
        not_full_.notify_one();

        // Return the item.
        return item;
    }


    // ============================================================
    // Close the queue.
    //
    // After close():
    //
    // - no new items can be pushed
    // - blocked producers wake up
    // - blocked consumers wake up
    // - consumers can still drain remaining items
    // ============================================================

    void close() {
        {
            std::lock_guard<std::mutex> lock(mutex_);

            closed_ = true;
        }

        not_empty_.notify_all();
        not_full_.notify_all();
    }


    // ============================================================
    // Queue inspection
    // queue_ may be modified by producer/consumer threads,
    // so mutex_ must be locked before reading queue_.size().
    // ============================================================

    // Return the current number of items in the queue.
    //
    // Because another thread may modify queue_,
    // mutex_ must be locked before reading queue_.size().
    //
    // Note:
    // This is a const function.
    std::size_t size() const {
        std::lock_guard<std::mutex> lock(mutex_);
        return queue_.size();
    }


    // Return the configured queue capacity.
    //
    // capacity_ does not change after construction.
    std::size_t capacity() const {
        return capacity_;
    }



private:
    // Stores queue elements.
    std::deque<T> queue_;


    // Maximum number of elements allowed.
    std::size_t capacity_;

    mutable std::mutex mutex_;


    std::condition_variable_any not_empty_;

    std::condition_variable_any not_full_;

    bool closed_;

};

Overwriting distributed-profiler/include/bounded_queue.hpp


## Fast Producer + Slow Consumer Baseline

### Goal

The goal of this section is to create the first controlled backpressure experiment.

We intentionally make the producer much faster than the consumer:

```text
Fast Producer
      ↓
BoundedQueue
      ↓
Slow Consumer
```

The purpose is to observe how a bounded queue behaves when downstream processing cannot keep up with event production.

---

## Experimental Configuration

We will start with a simple workload:

```text
Total events:
100

Queue capacity:
8

Producer delay:
0 ms

Consumer delay:
10 ms per event
```

The producer generates events as quickly as possible.

The consumer intentionally processes each event slowly.

This should cause the queue to fill quickly.

---

## Expected Behavior

At the beginning, the producer starts inserting events:

```text
Producer
    ↓
[A][B][C][D][ ][ ][ ][ ]
    ↓
Consumer
```

Because the producer is much faster than the consumer, the queue continues filling:

```text
Producer
    ↓
[A][B][C][D][E][F][G][ ]
    ↓
Consumer
```

Eventually the queue becomes full:

```text
Producer
    ↓
[A][B][C][D][E][F][G][H]
                         ↑
                        FULL
    ↓
Consumer
```

At this point, another call to:

```cpp
queue.push(...)
```

cannot immediately insert a new event.

The producer therefore waits inside the bounded queue.

Conceptually:

```text
Producer wants to push
        ↓
Queue full
        ↓
Producer waits
        ↓
Consumer pops one event
        ↓
Queue has free space
        ↓
Producer wakes up
        ↓
Producer pushes next event
```

This is the first direct demonstration of backpressure.

---

## Producer Responsibilities

The producer thread will:

```text
generate event
    ↓
increment attempted_events
    ↓
measure push start time
    ↓
queue.push(...)
    ↓
measure push end time
    ↓
increment produced_events
    ↓
update blocked-time measurement
    ↓
inspect queue occupancy
    ↓
update max_queue_occupancy
```

The producer does not intentionally sleep.

Therefore, its natural behavior is to push events as quickly as possible.

---

## Consumer Responsibilities

The consumer thread will:

```text
queue.pop()
    ↓
receive one event
    ↓
sleep for 10 ms
    ↓
increment consumed_events
    ↓
repeat
```

The artificial sleep represents slower downstream processing.

Later, the same type of delay can represent:

```text
serialization cost
network transmission
slow collector
database writes
analysis overhead
```

For this first experiment, however, we use only a simple sleep so that the bottleneck is easy to understand.

---

## Measuring Push Duration

Before calling:

```cpp
queue.push(...)
```

the producer records:

```cpp
auto push_start = Clock::now();
```

After the push returns:

```cpp
auto push_end = Clock::now();
```

Then:

```text
push_duration
=
push_end - push_start
```

If the queue is not full, this duration should be very small.

If the queue is full:

```text
push()
    ↓
wait
    ↓
consumer removes event
    ↓
push continues
```

then the duration becomes much larger.

For the baseline experiment, we use:

```text
push duration
≈
producer blocked time
```

This is an approximation because the measurement also includes:

```text
mutex acquisition
queue insertion
small function overhead
```

Later, we can instrument the queue internally to measure the actual waiting time more precisely.

---

## Measuring Queue Occupancy

After a successful push, we can inspect:

```cpp
queue.size()
```

and compare it with the previous maximum.

Conceptually:

```text
current_size = queue.size()

if current_size > max_queue_occupancy:
    max_queue_occupancy = current_size
```

For a queue with:

```text
capacity = 8
```

if we observe:

```text
max_queue_occupancy = 8
```

then the queue reached full capacity.

This indicates strong backpressure.

---

## Thread Lifecycle

The experiment uses two worker threads:

```text
Main Thread
│
├── Producer Thread
│
└── Consumer Thread
```

The producer generates all events and then closes the queue:

```text
produce all events
      ↓
queue.close()
```

Closing the queue means:

```text
no more events will be produced
```

but the consumer is still allowed to drain remaining items.

The consumer continues until:

```text
queue is empty
AND
queue is closed
```

Then:

```cpp
queue.pop(...)
```

returns:

```cpp
std::nullopt
```

and the consumer exits.

---

## Expected Thread Sequence

Conceptually:

```text
Main
  ↓
start producer
  ↓
start consumer

Producer                Consumer
   │                       │
   ├─ push                 ├─ pop
   ├─ push                 ├─ sleep
   ├─ push                 │
   ├─ push                 │
   ├─ queue full           │
   ├─ wait                 │
   │                       ├─ pop
   ├─ wake                 ├─ sleep
   ├─ push                 │
   │                       │
   ...                     ...
   │                       │
   ├─ finish producing     │
   └─ queue.close()        │
                           ├─ drain remaining events
                           └─ exit
```

The main thread waits for both worker threads to finish.

---

## Metrics Collected

The baseline experiment will collect:

```text
attempted_events
produced_events
consumed_events
dropped_events

max_queue_occupancy

producer_blocked_time

total_runtime
```

The blocking policy should produce:

```text
dropped_events = 0
```

because the producer waits instead of discarding events.

---

## Derived Metrics

After the experiment finishes, we calculate several derived metrics.

### Producer Throughput

```text
producer_throughput
=
produced_events
---------------
total_runtime
```

Unit:

```text
events / second
```

---

### Consumer Throughput

```text
consumer_throughput
=
consumed_events
---------------
total_runtime
```

Because the consumer is intentionally slow, it should determine the long-term pipeline throughput.

---

### Maximum Queue Utilization

```text
max_queue_utilization
=
max_queue_occupancy
-------------------
queue_capacity
× 100%
```

For example:

```text
max occupancy = 8
capacity = 8

utilization = 100%
```

This indicates that the queue became completely full.

---

### Producer Blocked Percentage

```text
blocked_percentage
=
producer_blocked_time
---------------------
total_runtime
× 100%
```

For example:

```text
runtime = 1.0 s

producer blocked time = 0.85 s

blocked percentage = 85%
```

This means the producer spends most of its execution time waiting for the consumer.

---

## Expected Result

Because the consumer sleeps:

```text
10 ms / event
```

its theoretical processing rate is roughly:

```text
100 events / second
```

The producer itself may be capable of generating events much faster than this.

However, after the queue becomes full, backpressure forces the producer to slow down.

Therefore, the effective producer throughput should eventually become close to the consumer throughput.

Conceptually:

```text
Producer raw capability
very high
      ↓
queue becomes full
      ↓
producer blocked
      ↓
effective producer throughput
≈
consumer throughput
```

---

## What This Experiment Demonstrates

The experiment demonstrates an important property of bounded pipelines:

```text
the slowest stage controls long-term throughput
```

A bounded queue prevents:

```text
unbounded memory growth
```

but does so by pushing pressure backward toward the producer.

The complete behavior is:

```text
Slow Consumer
      ↓
Queue fills
      ↓
Producer blocks
      ↓
Producer throughput decreases
```

This is backpressure propagation in its simplest form.

---

## Questions to Answer After This Experiment

After completing the baseline experiment, the following questions should be answerable:

- Why does the queue become full?
- Why does the producer block?
- Why does the consumer become the bottleneck?
- Why does producer throughput approach consumer throughput?
- How does a bounded queue prevent unlimited memory growth?
- What does `max_queue_occupancy == queue_capacity` indicate?
- How can push duration approximate producer blocked time?
- Why is this only an approximation?
- Why does closing the queue not immediately stop the consumer?
- Why should the consumer drain remaining events after `close()`?
- How does this experiment demonstrate backpressure propagation?

In [6]:
%%writefile distributed-profiler/benchmarks/backpressure_baseline.cpp

#include "bounded_queue.hpp"
#include "pipeline_metrics.hpp"
#include "profile_event.hpp"

#include <algorithm>
#include <chrono>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <stop_token>
#include <thread>


int main() {

    using Clock = std::chrono::steady_clock;


    // ============================================================
    // Experiment configuration
    // ============================================================

    const std::size_t total_events = 100;

    const std::size_t queue_capacity = 8;

    const auto consumer_delay =
        std::chrono::milliseconds(10);


    // ============================================================
    // Queue and metrics
    // ============================================================

    BoundedQueue<ProfileEvent> queue(queue_capacity);

    PipelineMetrics metrics;


    // ============================================================
    // Experiment start time
    // ============================================================

    auto experiment_start = Clock::now();


    // ============================================================
    // Producer thread
    //
    // create event
    //      ↓
    // queue.push()
    //      ↓
    // measure push duration
    //      ↓
    // inspect queue occupancy
    // ============================================================

    std::jthread producer(
        [&](std::stop_token stop_token) {

            for (
                std::size_t i = 0;
                i < total_events;
                ++i
            ) {
                // Producer attempts to generate one event.
                ++metrics.attempted_events;


                // Create a synthetic profiling event.
                ProfileEvent event =
                    KernelEvent{
                        static_cast<std::uint64_t>(i),
                        100,
                        0,
                        "synthetic_kernel"
                    };


                // Measure how long push() takes.
                auto push_start = Clock::now();

                bool pushed = queue.push(
                    std::move(event),
                    stop_token
                );

                auto push_end = Clock::now();


                // Duration includes:
                //
                // mutex acquisition
                // queue insertion
                // possible waiting due to full queue
                auto push_duration =
                    push_end - push_start;


                // If queue was closed or stop was requested,
                // stop producing.
                if (!pushed) {
                    break;
                }


                ++metrics.produced_events;


                // For this baseline:
                //
                // push duration ≈ producer blocked time
                metrics.producer_blocked_time +=
                    std::chrono::duration_cast<
                        std::chrono::nanoseconds
                    >(push_duration);


                // Inspect current queue occupancy.
                std::size_t current_size =
                    queue.size();


                // Update the maximum observed occupancy.
                metrics.max_queue_occupancy =
                    std::max(
                        metrics.max_queue_occupancy,
                        current_size
                    );
            }


            // No more events will be produced.
            queue.close();
        }
    );


    // ============================================================
    // Consumer thread
    //
    // queue.pop()
    //      ↓
    // simulate slow processing
    //      ↓
    // consumed_events++
    // ============================================================

    std::jthread consumer(
        [&](std::stop_token stop_token) {

            while (true) {

                auto event =
                    queue.pop(stop_token);


                // queue closed and completely drained,
                // or stop requested.
                if (!event.has_value()) {
                    break;
                }


                // Simulate a slow downstream stage.
                std::this_thread::sleep_for(
                    consumer_delay
                );


                ++metrics.consumed_events;
            }
        }
    );


    // ============================================================
    // Wait for worker threads
    // ============================================================

    producer.join();

    consumer.join();


    // ============================================================
    // Experiment end time
    // ============================================================

    auto experiment_end = Clock::now();

    metrics.total_runtime =
        std::chrono::duration_cast<
            std::chrono::nanoseconds
        >(
            experiment_end -
            experiment_start
        );


    // ============================================================
    // Derived metrics
    // ============================================================

    double runtime_seconds =
        std::chrono::duration<double>(
            metrics.total_runtime
        ).count();


    double blocked_seconds =
        std::chrono::duration<double>(
            metrics.producer_blocked_time
        ).count();


    double producer_throughput =
        static_cast<double>(
            metrics.produced_events
        )
        / runtime_seconds;


    double consumer_throughput =
        static_cast<double>(
            metrics.consumed_events
        )
        / runtime_seconds;


    double max_queue_utilization =
        100.0 *
        static_cast<double>(
            metrics.max_queue_occupancy
        )
        /
        static_cast<double>(
            queue.capacity()
        );


    double producer_blocked_percentage =
        100.0 *
        blocked_seconds
        /
        runtime_seconds;


    // ============================================================
    // Print experiment results
    // ============================================================

    std::cout
        << "\n=== Backpressure Baseline ===\n";


    // Workload metrics
    std::cout
        << "Attempted events: "
        << metrics.attempted_events
        << '\n';

    std::cout
        << "Produced events: "
        << metrics.produced_events
        << '\n';

    std::cout
        << "Consumed events: "
        << metrics.consumed_events
        << '\n';

    std::cout
        << "Dropped events: "
        << metrics.dropped_events
        << '\n';


    // Queue metrics
    std::cout
        << "\nQueue capacity: "
        << queue.capacity()
        << '\n';

    std::cout
        << "Max queue occupancy: "
        << metrics.max_queue_occupancy
        << '\n';

    std::cout
        << "Max queue utilization: "
        << max_queue_utilization
        << " %\n";


    // Timing metrics
    std::cout
        << "\nTotal runtime: "
        << runtime_seconds
        << " s\n";

    std::cout
        << "Producer push/block time: "
        << blocked_seconds
        << " s\n";

    std::cout
        << "Producer blocked percentage: "
        << producer_blocked_percentage
        << " %\n";


    // Throughput metrics
    std::cout
        << "\nProducer throughput: "
        << producer_throughput
        << " events/s\n";

    std::cout
        << "Consumer throughput: "
        << consumer_throughput
        << " events/s\n";


    return 0;
}

Writing distributed-profiler/benchmarks/backpressure_baseline.cpp


In [7]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/benchmarks/backpressure_baseline.cpp \
    -pthread \
    -o backpressure_baseline

In [8]:
!./backpressure_baseline


=== Backpressure Baseline ===
Attempted events: 100
Produced events: 100
Consumed events: 100
Dropped events: 0

Queue capacity: 8
Max queue occupancy: 8
Max queue utilization: 100 %

Total runtime: 1.01631 s
Producer push/block time: 0.92454 s
Producer blocked percentage: 90.9704 %

Producer throughput: 98.3954 events/s
Consumer throughput: 98.3954 events/s


## Precise Producer Blocking Metrics

### Goal

The previous baseline experiment showed clear backpressure:

```text
Queue capacity reached 100%

Producer spent most of the runtime inside queue.push()

Producer throughput was forced down to consumer throughput
```

However, the previous measurement used:

```text
push duration
≈
producer blocked time
```

This is only an approximation.

The measured `push()` duration includes:

```text
mutex acquisition
condition-variable waiting
queue insertion
small function overhead
```

What we really want to measure is:

> How many times did the producer actually have to wait because the queue was full, and how long did it really wait?

---

## Why More Precise Instrumentation Is Needed

In the baseline implementation, we measured:

```cpp
auto push_start = Clock::now();

queue.push(...);

auto push_end = Clock::now();
```

Then:

```text
push_duration = push_end - push_start
```

If the queue is not full, `push_duration` may still be non-zero because `push()` performs:

```text
mutex lock
predicate check
queue insertion
mutex unlock
notification
```

Therefore:

```text
push_duration
```

is not exactly the same as:

```text
time blocked by backpressure
```

For more accurate analysis, the queue itself should detect when it actually has to wait.

---

## Desired Information from `push()`

For every push operation, we want to know:

```text
Was the push successful?

Did the producer encounter a full queue?

How long did it wait for space?
```

A useful result structure is:

```cpp
struct PushResult {
    bool success = false;
    bool blocked = false;

    std::chrono::nanoseconds
        wait_time{0};
};
```

Then:

```text
success
```

tells us whether the event entered the queue.

```text
blocked
```

tells us whether the queue was full when the producer attempted to push.

```text
wait_time
```

tells us how long the producer had to wait for free capacity.

---

## Updated Push Flow

The new `push()` behavior becomes:

```text
producer calls push()
        ↓
lock mutex
        ↓
is queue full?
   ┌────┴────┐
   │         │
  no        yes
   │         │
   │      blocked = true
   │         ↓
   │      start timer
   │         ↓
   │      wait()
   │         ↓
   │      space available
   │         ↓
   │      stop timer
   │         ↓
   │      calculate wait_time
   │
   └─────────┬─────────
             ↓
       push item
             ↓
       return PushResult
```

---

## Detecting Whether the Producer Blocks

Before calling `wait()`, the queue can inspect:

```cpp
queue_.size() >= capacity_
```

If this is true, then the producer has encountered backpressure.

Conceptually:

```cpp
bool blocked =
    queue_.size() >= capacity_;
```

If:

```text
blocked == false
```

then the item can normally be inserted without waiting for capacity.

If:

```text
blocked == true
```

then the producer must wait until the consumer removes an item.

---

## Measuring Actual Wait Time

If the queue is full:

```text
queue full
    ↓
start wait timer
    ↓
condition-variable wait
    ↓
consumer pops event
    ↓
producer wakes
    ↓
stop wait timer
```

Conceptually:

```cpp
auto wait_start = Clock::now();

not_full_.wait(...);

auto wait_end = Clock::now();
```

Then:

```cpp
wait_time =
    wait_end - wait_start;
```

This is much closer to the actual backpressure delay than timing the entire `push()` call externally.

---

## New Producer Metrics

With `PushResult`, the producer can update the metrics directly:

```text
attempt push
    ↓
PushResult
    ↓
success?
    ↓ yes
produced_events++
```

and:

```text
blocked?
    ↓ yes
producer_block_count++
producer_blocked_time += wait_time
```

This gives us two useful backpressure metrics:

```text
producer_block_count
producer_blocked_time
```

---

## Producer Block Count

`producer_block_count` tells us:

> How many push operations encountered a full queue?

For example:

```text
Total pushes:
100

Blocked pushes:
91
```

This means most pushes occurred while the pipeline was already under backpressure.

---

## Producer Blocked Time

`producer_blocked_time` tells us:

> How much total time did the producer spend waiting for queue capacity?

For example:

```text
Total runtime:
1.0 s

Producer blocked time:
0.90 s
```

This indicates that the producer spent almost the entire experiment waiting for the consumer.

---

## Average Blocking Time

Once we have:

```text
producer_block_count
producer_blocked_time
```

we can calculate:

```text
average_block_time
=
producer_blocked_time
---------------------
producer_block_count
```

For example:

```text
blocked time = 900 ms
block count = 90

average block time = 10 ms
```

If the consumer processes approximately one event every:

```text
10 ms
```

then this result would make sense.

---

## Relationship to Consumer Delay

Suppose:

```text
consumer delay = 10 ms/event
```

and the queue remains full.

Then a blocked producer usually needs to wait until:

```text
consumer finishes one event
        ↓
consumer pops another event
        ↓
one queue slot becomes available
```

Therefore, once the system reaches steady-state backpressure, the producer's waiting time per blocked push may be close to the consumer's processing interval.

Conceptually:

```text
Consumer frees ~1 slot every 10 ms

Producer waits for one slot

Average producer wait
≈ 10 ms
```

---

## Updated Metrics Structure

The relevant metrics now become:

```text
PipelineMetrics
│
├── attempted_events
├── produced_events
├── consumed_events
├── dropped_events
│
├── producer_block_count
├── producer_blocked_time
│
├── max_queue_occupancy
│
└── total_runtime
```

From these values, we can derive:

```text
producer throughput

consumer throughput

blocked percentage

average block time

queue utilization
```

---

## Derived Backpressure Metrics

### Blocked Percentage

```text
blocked_percentage
=
producer_blocked_time
---------------------
total_runtime
× 100%
```

This tells us what fraction of the experiment the producer spent waiting.

---

### Average Block Time

```text
average_block_time
=
producer_blocked_time
---------------------
producer_block_count
```

This tells us how expensive each backpressure event is on average.

---

### Blocked Push Ratio

We can also calculate:

```text
blocked_push_ratio
=
producer_block_count
--------------------
produced_events
× 100%
```

For example:

```text
produced events = 100
blocked pushes = 91

blocked push ratio = 91%
```

This shows how frequently event production encounters a full queue.

---

## Why This Matters for Profiling

For a profiling system, producer blocking may directly affect the application being measured.

Conceptually:

```text
Application thread
       ↓
record profiling event
       ↓
queue full
       ↓
wait
       ↓
application delayed
```

Therefore, precise blocking metrics help answer:

> How much overhead is the profiling infrastructure adding to the application?

This is more useful than only measuring total throughput.

---

## Expected Result

For the current workload:

```text
Total events:
100

Queue capacity:
8

Consumer delay:
10 ms
```

we expect:

```text
max queue occupancy:
8

queue utilization:
100%

producer block count:
high

producer blocked time:
high

average blocked time:
close to consumer processing interval
```

The exact numbers depend on thread scheduling, but the overall pattern should be clear.

---

## Why We Do This Before the Queue Capacity Benchmark

The next major experiment will compare:

```text
capacity = 1
capacity = 8
capacity = 64
capacity = 256
capacity = 1024
```

To compare those configurations meaningfully, we want accurate measurements of:

```text
how often the producer blocks

how long the producer blocks
```

Therefore, precise queue instrumentation should be added before the capacity experiment.

---

## Questions to Answer After This Section

After completing this section, the following questions should be answerable:

- Why is timing the entire `push()` only an approximation of blocking time?
- How can the queue determine whether a push actually encounters backpressure?
- Why should blocking time be measured inside the queue?
- What does `producer_block_count` represent?
- What does `producer_blocked_time` represent?
- How can average block time be calculated?
- Why might average block time be close to the consumer processing delay?
- Why are these metrics important for profiling overhead?
- Why should precise instrumentation be added before comparing queue capacities?

In [9]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <chrono>
#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <stop_token>
#include <utility>


// ============================================================
// Result returned by push().
//
// success:
//     Was the item successfully inserted?
//
// blocked:
//     Was the queue full when push() arrived?
//
// wait_time:
//     How long did the producer wait for queue capacity?
// ============================================================

struct PushResult {
    bool success = false;
    bool blocked = false;

    std::chrono::nanoseconds wait_time{0};
};


template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity)
        : capacity_(capacity),
          closed_(false) {}


    // ============================================================
    // Push one item into the queue.
    //
    // In addition to inserting the item, this function now
    // measures actual backpressure waiting time.
    // ============================================================

    PushResult push(
        T item,
        std::stop_token stop_token
    ) {
        using Clock = std::chrono::steady_clock;

        std::unique_lock<std::mutex> lock(mutex_);

        PushResult result;


        // --------------------------------------------------------
        // Check whether the queue is already full.
        //
        // If it is full at this point, this push operation
        // encounters backpressure.
        // --------------------------------------------------------

        result.blocked =
            queue_.size() >= capacity_;


        // --------------------------------------------------------
        // Start timing only if the producer actually needs to wait.
        // --------------------------------------------------------

        Clock::time_point wait_start;

        if (result.blocked) {
            wait_start = Clock::now();
        }


        // --------------------------------------------------------
        // Wait until:
        //
        // - queue has free capacity
        // - queue is closed
        // - stop is requested
        //
        // condition_variable_any releases mutex_ while sleeping
        // and reacquires it before returning.
        // --------------------------------------------------------

        bool ready = not_full_.wait(
            lock,
            stop_token,
            [this]() {
                return queue_.size() < capacity_
                    || closed_;
            }
        );


        // --------------------------------------------------------
        // If we actually blocked, stop the wait timer now.
        // --------------------------------------------------------

        if (result.blocked) {
            auto wait_end = Clock::now();

            result.wait_time =
                std::chrono::duration_cast<
                    std::chrono::nanoseconds
                >(
                    wait_end - wait_start
                );
        }


        // --------------------------------------------------------
        // Push cannot continue if:
        //
        // - wait was interrupted
        // - stop requested
        // - queue closed
        // --------------------------------------------------------

        if (
            !ready ||
            stop_token.stop_requested() ||
            closed_
        ) {
            result.success = false;
            return result;
        }


        // --------------------------------------------------------
        // Insert item into queue.
        // --------------------------------------------------------

        queue_.push_back(std::move(item));

        result.success = true;


        // Release mutex before notification.
        lock.unlock();

        not_empty_.notify_one();

        return result;
    }


    // ============================================================
    // Pop one item.
    // ============================================================

    std::optional<T> pop(
        std::stop_token stop_token
    ) {
        std::unique_lock<std::mutex> lock(mutex_);

        bool ready = not_empty_.wait(
            lock,
            stop_token,
            [this]() {
                return !queue_.empty()
                    || closed_;
            }
        );

        if (
            !ready ||
            stop_token.stop_requested()
        ) {
            return std::nullopt;
        }

        if (
            queue_.empty() &&
            closed_
        ) {
            return std::nullopt;
        }

        T item =
            std::move(queue_.front());

        queue_.pop_front();

        lock.unlock();

        not_full_.notify_one();

        return item;
    }


    // ============================================================
    // Close queue.
    // ============================================================

    void close() {
        {
            std::lock_guard<std::mutex> lock(mutex_);

            closed_ = true;
        }

        not_empty_.notify_all();
        not_full_.notify_all();
    }


    // ============================================================
    // Current queue occupancy.
    // ============================================================

    std::size_t size() const {
        std::lock_guard<std::mutex> lock(mutex_);

        return queue_.size();
    }


    // ============================================================
    // Maximum queue capacity.
    // ============================================================

    std::size_t capacity() const {
        return capacity_;
    }


private:
    std::deque<T> queue_;

    std::size_t capacity_;

    mutable std::mutex mutex_;

    std::condition_variable_any not_empty_;

    std::condition_variable_any not_full_;

    bool closed_;
};

Overwriting distributed-profiler/include/bounded_queue.hpp


In [10]:
%%writefile distributed-profiler/benchmarks/backpressure_baseline.cpp

#include "bounded_queue.hpp"
#include "pipeline_metrics.hpp"
#include "profile_event.hpp"

#include <algorithm>
#include <chrono>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <stop_token>
#include <thread>


int main() {

    using Clock = std::chrono::steady_clock;


    // ============================================================
    // Experiment configuration
    // ============================================================

    const std::size_t total_events = 100;

    const std::size_t queue_capacity = 8;

    const auto consumer_delay =
        std::chrono::milliseconds(10);


    // ============================================================
    // Queue and metrics
    // ============================================================

    BoundedQueue<ProfileEvent> queue(
        queue_capacity
    );

    PipelineMetrics metrics;


    // ============================================================
    // Start experiment timer.
    // ============================================================

    auto experiment_start =
        Clock::now();


    // ============================================================
    // Producer
    // ============================================================

    std::jthread producer(
        [&](std::stop_token stop_token) {

            for (
                std::size_t i = 0;
                i < total_events;
                ++i
            ) {

                ++metrics.attempted_events;


                ProfileEvent event =
                    KernelEvent{
                        static_cast<std::uint64_t>(i),
                        100,
                        0,
                        "synthetic_kernel"
                    };


                // ------------------------------------------------
                // push() now returns detailed backpressure info.
                // ------------------------------------------------

                PushResult result =
                    queue.push(
                        std::move(event),
                        stop_token
                    );


                // ------------------------------------------------
                // Stop if insertion failed.
                // ------------------------------------------------

                if (!result.success) {
                    break;
                }


                ++metrics.produced_events;


                // ------------------------------------------------
                // Record precise blocking metrics.
                //
                // Only count this push if it actually encountered
                // a full queue.
                // ------------------------------------------------

                if (result.blocked) {

                    ++metrics.producer_block_count;

                    metrics.producer_blocked_time +=
                        result.wait_time;
                }


                // ------------------------------------------------
                // Observe queue occupancy.
                // ------------------------------------------------

                std::size_t current_size =
                    queue.size();

                metrics.max_queue_occupancy =
                    std::max(
                        metrics.max_queue_occupancy,
                        current_size
                    );
            }


            // Signal that no more events will arrive.
            queue.close();
        }
    );


    // ============================================================
    // Consumer
    // ============================================================

    std::jthread consumer(
        [&](std::stop_token stop_token) {

            while (true) {

                auto event =
                    queue.pop(stop_token);


                if (!event.has_value()) {
                    break;
                }


                // Simulate slow downstream processing.
                std::this_thread::sleep_for(
                    consumer_delay
                );


                ++metrics.consumed_events;
            }
        }
    );


    // ============================================================
    // Wait for both threads.
    // ============================================================

    producer.join();

    consumer.join();


    // ============================================================
    // Stop experiment timer.
    // ============================================================

    auto experiment_end =
        Clock::now();


    metrics.total_runtime =
        std::chrono::duration_cast<
            std::chrono::nanoseconds
        >(
            experiment_end -
            experiment_start
        );


    // ============================================================
    // Convert timing values.
    // ============================================================

    double runtime_seconds =
        std::chrono::duration<double>(
            metrics.total_runtime
        ).count();


    double blocked_seconds =
        std::chrono::duration<double>(
            metrics.producer_blocked_time
        ).count();


    // ============================================================
    // Throughput
    // ============================================================

    double producer_throughput =
        static_cast<double>(
            metrics.produced_events
        )
        / runtime_seconds;


    double consumer_throughput =
        static_cast<double>(
            metrics.consumed_events
        )
        / runtime_seconds;


    // ============================================================
    // Queue utilization
    // ============================================================

    double max_queue_utilization =
        100.0 *
        static_cast<double>(
            metrics.max_queue_occupancy
        )
        /
        static_cast<double>(
            queue.capacity()
        );


    // ============================================================
    // Blocking percentage
    // ============================================================

    double producer_blocked_percentage =
        100.0 *
        blocked_seconds
        /
        runtime_seconds;


    // ============================================================
    // Average block time
    // ============================================================

    double average_block_ms = 0.0;

    if (metrics.producer_block_count > 0) {

        average_block_ms =
            std::chrono::duration<double, std::milli>(
                metrics.producer_blocked_time
            ).count()
            /
            static_cast<double>(
                metrics.producer_block_count
            );
    }


    // ============================================================
    // Blocked push ratio
    // ============================================================

    double blocked_push_ratio = 0.0;

    if (metrics.produced_events > 0) {

        blocked_push_ratio =
            100.0 *
            static_cast<double>(
                metrics.producer_block_count
            )
            /
            static_cast<double>(
                metrics.produced_events
            );
    }


    // ============================================================
    // Print results
    // ============================================================

    std::cout
        << "\n=== Precise Backpressure Metrics ===\n";


    // Workload
    std::cout
        << "Attempted events: "
        << metrics.attempted_events
        << '\n';

    std::cout
        << "Produced events: "
        << metrics.produced_events
        << '\n';

    std::cout
        << "Consumed events: "
        << metrics.consumed_events
        << '\n';

    std::cout
        << "Dropped events: "
        << metrics.dropped_events
        << '\n';


    // Queue
    std::cout
        << "\nQueue capacity: "
        << queue.capacity()
        << '\n';

    std::cout
        << "Max queue occupancy: "
        << metrics.max_queue_occupancy
        << '\n';

    std::cout
        << "Max queue utilization: "
        << max_queue_utilization
        << " %\n";


    // Backpressure
    std::cout
        << "\nProducer block count: "
        << metrics.producer_block_count
        << '\n';

    std::cout
        << "Producer blocked time: "
        << blocked_seconds
        << " s\n";

    std::cout
        << "Producer blocked percentage: "
        << producer_blocked_percentage
        << " %\n";

    std::cout
        << "Average block time: "
        << average_block_ms
        << " ms\n";

    std::cout
        << "Blocked push ratio: "
        << blocked_push_ratio
        << " %\n";


    // Throughput
    std::cout
        << "\nTotal runtime: "
        << runtime_seconds
        << " s\n";

    std::cout
        << "Producer throughput: "
        << producer_throughput
        << " events/s\n";

    std::cout
        << "Consumer throughput: "
        << consumer_throughput
        << " events/s\n";


    return 0;
}

Overwriting distributed-profiler/benchmarks/backpressure_baseline.cpp


In [11]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/benchmarks/backpressure_baseline.cpp \
    -pthread \
    -o backpressure_baseline

In [12]:
!./backpressure_baseline


=== Precise Backpressure Metrics ===
Attempted events: 100
Produced events: 100
Consumed events: 100
Dropped events: 0

Queue capacity: 8
Max queue occupancy: 8
Max queue utilization: 100 %

Producer block count: 92
Producer blocked time: 0.924844 s
Producer blocked percentage: 90.939 %
Average block time: 10.0527 ms
Blocked push ratio: 92 %

Total runtime: 1.01699 s
Producer throughput: 98.329 events/s
Consumer throughput: 98.329 events/s


## Queue Capacity Experiment

### Goal

The goal of this section is to study how the capacity of a bounded queue affects backpressure.

We will keep the workload unchanged and vary only:

```text
queue capacity
```

This allows us to isolate the effect of buffering.

The central question is:

> If the producer is much faster than the consumer, how does a larger queue change producer blocking and pipeline behavior?

---

## Fixed Workload

We will keep the following parameters constant:

```text
Total events:
100

Producer delay:
0 ms

Consumer delay:
10 ms per event
```

The consumer therefore remains the bottleneck.

Its approximate processing rate is:

```text
1 event / 10 ms
≈
100 events / second
```

The producer generates events much faster than that.

---

## Queue Capacities

We will compare several queue capacities:

```text
1
8
16
32
64
128
```

Each experiment uses exactly the same producer and consumer workload.

Only the amount of buffering changes.

---

## Experimental Structure

Instead of writing a separate program for every queue size, we will refactor the baseline experiment into a reusable function.

Conceptually:

```text
run_experiment(queue_capacity)
        ↓
create queue
        ↓
create producer
        ↓
create consumer
        ↓
run workload
        ↓
collect metrics
        ↓
return result
```

Then:

```text
capacity = 1
    ↓
run experiment

capacity = 8
    ↓
run experiment

capacity = 16
    ↓
run experiment

...
```

Finally, we compare the results in one table.

---

## Why Refactor Into `run_experiment()`

The previous benchmark placed the entire experiment inside:

```cpp
main()
```

That worked for one configuration.

Now we want to repeat the same workload many times.

A reusable function makes this easier:

```cpp
ExperimentResult run_experiment(
    std::size_t queue_capacity
);
```

Conceptually:

```text
input:
queue capacity

output:
metrics for that experiment
```

This also prepares the benchmark for later experiments involving:

```text
batch size
consumer delay
overload policy
```

---

## Expected Behavior with a Small Queue

Consider:

```text
capacity = 1
```

The queue can store only one event.

The sequence will look roughly like:

```text
Producer pushes event
        ↓
queue full
        ↓
producer tries next event
        ↓
producer blocks
        ↓
consumer removes event
        ↓
producer continues
```

The producer will therefore encounter backpressure very frequently.

We expect:

```text
high block count
high blocked push ratio
high blocked time
```

---

## Expected Behavior with Capacity 8

This is similar to the baseline experiment:

```text
capacity = 8
```

The producer can initially push several events quickly:

```text
[1][2][3][4][5][6][7][8]
```

After the queue becomes full:

```text
producer blocks
```

The consumer then effectively controls how quickly new events can enter the queue.

We already observed approximately:

```text
92 blocked pushes out of 100
```

because:

```text
100 total events
-
8 initially buffered events
≈
92 blocked events
```

---

## Expected Behavior with a Larger Queue

Consider:

```text
capacity = 64
```

The producer can initially buffer many more events:

```text
Producer
   ↓
[1][2][3] ... [64]
```

Only after the queue fills does backpressure reach the producer.

Therefore, compared with a queue of capacity 8, we expect:

```text
fewer blocked pushes
less total producer blocked time
producer finishes earlier
```

However, the consumer still requires approximately:

```text
10 ms per event
```

so the total time required to consume all 100 events remains close to:

```text
1 second
```

---

## Capacity Larger Than the Entire Workload

Consider:

```text
capacity = 128
```

but:

```text
total events = 100
```

The entire workload can fit inside the queue.

Conceptually:

```text
Producer
    ↓
push all 100 events rapidly
    ↓
finish
    ↓
queue contains many pending events
    ↓
Consumer slowly drains them
```

The producer may experience almost no backpressure.

However:

```text
consumer throughput does not improve
```

The total pipeline runtime is still dominated by the consumer.

---

## Important Distinction

A larger queue can improve:

```text
producer blocking
producer completion time
short-term burst absorption
```

but it does not make the slow consumer faster.

Therefore:

```text
larger queue
!=
higher downstream processing capacity
```

The queue changes buffering behavior, not the bottleneck itself.

---

## Expected Relationship

The expected trend is:

```text
Queue capacity increases
        ↓
Producer can buffer more events
        ↓
Backpressure reaches producer later
        ↓
Block count decreases
        ↓
Producer blocked time decreases
```

However:

```text
Consumer rate remains unchanged
        ↓
Long-term pipeline throughput remains similar
```

---

## Metrics to Compare

For each queue capacity, we will collect:

```text
producer_block_count

producer_blocked_time

average_block_time

blocked_push_ratio

max_queue_occupancy

max_queue_utilization

producer_throughput

consumer_throughput

total_runtime
```

---

## Expected Results Table

The benchmark will eventually produce a table similar to:

| Queue Capacity | Block Count | Blocked Push Ratio | Blocked Time | Max Occupancy | Runtime |
|---:|---:|---:|---:|---:|---:|
| 1 | ... | ... | ... | ... | ... |
| 8 | ... | ... | ... | ... | ... |
| 16 | ... | ... | ... | ... | ... |
| 32 | ... | ... | ... | ... | ... |
| 64 | ... | ... | ... | ... | ... |
| 128 | ... | ... | ... | ... | ... |

The most important trends should be easier to see across multiple runs than in a single baseline experiment.

---

## Expected Backpressure Trend

Conceptually:

```text
Capacity = 1

Producer
   ↓
[FULL]
   ↓
block almost immediately
```

```text
Capacity = 8

Producer
   ↓
[........]
   ↓
block after small burst
```

```text
Capacity = 64

Producer
   ↓
[........................................................]
   ↓
block much later
```

```text
Capacity = 128

Producer
   ↓
entire workload fits
   ↓
little or no producer blocking
```

---

## Queue Capacity Tradeoff

A larger queue reduces immediate backpressure, but it is not free.

### Small Queue

Advantages:

```text
low memory usage
small backlog
lower potential queueing delay
```

Disadvantages:

```text
frequent producer blocking
higher application interference
poor burst absorption
```

### Large Queue

Advantages:

```text
absorbs bursts
reduces short-term producer blocking
allows producer to finish sooner
```

Disadvantages:

```text
higher memory usage
larger backlog
potentially higher event latency
```

Therefore:

```text
larger queue
!=
always better
```

---

## Profiling-System Interpretation

For a profiling system, this tradeoff matters because the producer may run on or near the application thread.

A very small queue may cause:

```text
Application
    ↓
record event
    ↓
queue full
    ↓
block
    ↓
application delayed
```

A larger queue can reduce this immediate interference:

```text
Application
    ↓
record event
    ↓
buffer event
    ↓
continue
```

but now more profiling data may remain buffered for longer periods.

So queue capacity represents a tradeoff between:

```text
application interference
        vs
memory usage
        vs
event latency
```

---

## What We Expect to Learn

This experiment should demonstrate that queue capacity mainly controls:

```text
how much burst traffic can be absorbed
```

and:

```text
how quickly backpressure reaches the producer
```

It does not fundamentally change:

```text
consumer processing capability
```

If the consumer remains the bottleneck, increasing queue capacity only delays the moment when backpressure reaches the producer.

---

## Questions to Answer After This Experiment

After completing this section, the following questions should be answerable:

- How does queue capacity affect producer blocking?
- Why does a larger queue reduce the blocked push count?
- Why might total pipeline runtime remain almost unchanged?
- Why does a larger queue not make the consumer faster?
- What happens when queue capacity is larger than the entire workload?
- What is burst absorption?
- Why can a larger queue increase event latency?
- Why is queue capacity a tradeoff between memory, latency, and producer interference?
- Why should queue capacity be treated as a tuning parameter rather than simply maximized?

In [13]:
%%writefile distributed-profiler/benchmarks/queue_capacity_benchmark.cpp

#include "bounded_queue.hpp"
#include "pipeline_metrics.hpp"
#include "profile_event.hpp"

#include <algorithm>
#include <chrono>
#include <cstddef>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <stop_token>
#include <thread>
#include <vector>


struct ExperimentResult {
    std::size_t queue_capacity = 0;

    PipelineMetrics metrics;

    double runtime_seconds = 0.0;
    double producer_throughput = 0.0;
    double consumer_throughput = 0.0;

    double blocked_seconds = 0.0;
    double blocked_percentage = 0.0;
    double average_block_ms = 0.0;
    double blocked_push_ratio = 0.0;

    double max_queue_utilization = 0.0;
};


// ============================================================
// Run one producer-consumer experiment with a given
// queue capacity.
//
// Fixed workload:
//
// total_events   = 100
// consumer_delay = 10 ms
//
// Only queue_capacity changes.
// ============================================================

ExperimentResult run_experiment(
    std::size_t queue_capacity,
    std::size_t total_events,
    std::chrono::milliseconds consumer_delay
) {
    using Clock = std::chrono::steady_clock;

    BoundedQueue<ProfileEvent> queue(
        queue_capacity
    );

    PipelineMetrics metrics;


    // ============================================================
    // Start experiment timer.
    // ============================================================

    auto experiment_start =
        Clock::now();


    // ============================================================
    // Producer
    //
    // Producer generates events as fast as possible.
    //
    // If the queue becomes full:
    //
    // producer
    //    ↓
    // push()
    //    ↓
    // wait for free queue capacity
    //
    // PushResult tells us:
    //
    // success
    // blocked
    // wait_time
    // ============================================================

    std::jthread producer(
        [&](std::stop_token stop_token) {

            for (
                std::size_t i = 0;
                i < total_events;
                ++i
            ) {
                ++metrics.attempted_events;


                ProfileEvent event =
                    KernelEvent{
                        static_cast<std::uint64_t>(i),
                        100,
                        0,
                        "synthetic_kernel"
                    };


                PushResult result =
                    queue.push(
                        std::move(event),
                        stop_token
                    );


                if (!result.success) {
                    break;
                }


                ++metrics.produced_events;


                // ------------------------------------------------
                // Record precise backpressure metrics.
                // ------------------------------------------------

                if (result.blocked) {

                    ++metrics.producer_block_count;

                    metrics.producer_blocked_time +=
                        result.wait_time;
                }


                // ------------------------------------------------
                // Observe queue occupancy.
                //
                // queue.size() gives a thread-safe snapshot.
                // ------------------------------------------------

                std::size_t current_size =
                    queue.size();


                metrics.max_queue_occupancy =
                    std::max(
                        metrics.max_queue_occupancy,
                        current_size
                    );
            }


            // No more events will be produced.
            queue.close();
        }
    );


    // ============================================================
    // Consumer
    //
    // Consumer intentionally processes events slowly.
    //
    // queue.pop()
    //      ↓
    // sleep 10 ms
    //      ↓
    // consumed_events++
    //
    // This makes the consumer the bottleneck.
    // ============================================================

    std::jthread consumer(
        [&](std::stop_token stop_token) {

            while (true) {

                auto event =
                    queue.pop(stop_token);


                if (!event.has_value()) {
                    break;
                }


                std::this_thread::sleep_for(
                    consumer_delay
                );


                ++metrics.consumed_events;
            }
        }
    );


    // ============================================================
    // Wait for both threads to finish.
    // ============================================================

    producer.join();

    consumer.join();


    // ============================================================
    // Stop experiment timer.
    // ============================================================

    auto experiment_end =
        Clock::now();


    metrics.total_runtime =
        std::chrono::duration_cast<
            std::chrono::nanoseconds
        >(
            experiment_end -
            experiment_start
        );


    // ============================================================
    // Derived metrics
    // ============================================================

    double runtime_seconds =
        std::chrono::duration<double>(
            metrics.total_runtime
        ).count();


    double blocked_seconds =
        std::chrono::duration<double>(
            metrics.producer_blocked_time
        ).count();


    double producer_throughput =
        static_cast<double>(
            metrics.produced_events
        )
        / runtime_seconds;


    double consumer_throughput =
        static_cast<double>(
            metrics.consumed_events
        )
        / runtime_seconds;


    double blocked_percentage =
        100.0 *
        blocked_seconds
        /
        runtime_seconds;


    double max_queue_utilization =
        100.0 *
        static_cast<double>(
            metrics.max_queue_occupancy
        )
        /
        static_cast<double>(
            queue_capacity
        );


    double average_block_ms = 0.0;

    if (
        metrics.producer_block_count > 0
    ) {
        average_block_ms =
            std::chrono::duration<
                double,
                std::milli
            >(
                metrics.producer_blocked_time
            ).count()
            /
            static_cast<double>(
                metrics.producer_block_count
            );
    }


    double blocked_push_ratio = 0.0;

    if (
        metrics.produced_events > 0
    ) {
        blocked_push_ratio =
            100.0 *
            static_cast<double>(
                metrics.producer_block_count
            )
            /
            static_cast<double>(
                metrics.produced_events
            );
    }


    // ============================================================
    // Package everything into ExperimentResult.
    // ============================================================

    ExperimentResult result;

    result.queue_capacity =
        queue_capacity;

    result.metrics =
        metrics;

    result.runtime_seconds =
        runtime_seconds;

    result.producer_throughput =
        producer_throughput;

    result.consumer_throughput =
        consumer_throughput;

    result.blocked_seconds =
        blocked_seconds;

    result.blocked_percentage =
        blocked_percentage;

    result.average_block_ms =
        average_block_ms;

    result.blocked_push_ratio =
        blocked_push_ratio;

    result.max_queue_utilization =
        max_queue_utilization;


    return result;
}


int main() {

    // ============================================================
    // Fixed workload.
    // ============================================================

    const std::size_t total_events =
        100;

    const auto consumer_delay =
        std::chrono::milliseconds(10);


    // ============================================================
    // Queue capacities to compare.
    // ============================================================

    const std::vector<std::size_t> capacities = {
        1,
        8,
        16,
        32,
        64,
        128
    };


    std::vector<ExperimentResult> results;

    results.reserve(
        capacities.size()
    );


    // ============================================================
    // Run exactly the same workload for every queue capacity.
    //
    // Queue capacity is the only variable.
    // ============================================================

    for (
        std::size_t capacity :
        capacities
    ) {

        ExperimentResult result =
            run_experiment(
                capacity,
                total_events,
                consumer_delay
            );


        results.push_back(
            std::move(result)
        );
    }


    // ============================================================
    // Print comparison table.
    // ============================================================

    std::cout
        << "\n=== Queue Capacity Experiment ===\n\n";


    std::cout
        << std::left
        << std::setw(10)
        << "Capacity"

        << std::setw(12)
        << "Blocks"

        << std::setw(14)
        << "Block Ratio"

        << std::setw(14)
        << "Block Time"

        << std::setw(14)
        << "Avg Block"

        << std::setw(12)
        << "Max Queue"

        << std::setw(14)
        << "Queue Util"

        << std::setw(12)
        << "Runtime"

        << '\n';


    std::cout
        << std::string(102, '-')
        << '\n';


    for (
        const auto& result :
        results
    ) {

        std::cout
            << std::left

            << std::setw(10)
            << result.queue_capacity

            << std::setw(12)
            << result.metrics.producer_block_count

            << std::setw(14)
            << (
                std::to_string(
                    result.blocked_push_ratio
                ).substr(0, 5)
                + "%"
            )

            << std::setw(14)
            << result.blocked_seconds

            << std::setw(14)
            << result.average_block_ms

            << std::setw(12)
            << result.metrics.max_queue_occupancy

            << std::setw(14)
            << (
                std::to_string(
                    result.max_queue_utilization
                ).substr(0, 6)
                + "%"
            )

            << std::setw(12)
            << result.runtime_seconds

            << '\n';
    }


    // ============================================================
    // Print throughput separately.
    //
    // We expect throughput to remain similar because the
    // consumer is still the bottleneck.
    // ============================================================

    std::cout
        << "\nThroughput comparison\n\n";


    std::cout
        << std::left
        << std::setw(12)
        << "Capacity"

        << std::setw(22)
        << "Producer events/s"

        << std::setw(22)
        << "Consumer events/s"

        << '\n';


    std::cout
        << std::string(56, '-')
        << '\n';


    for (
        const auto& result :
        results
    ) {

        std::cout
            << std::left

            << std::setw(12)
            << result.queue_capacity

            << std::setw(22)
            << result.producer_throughput

            << std::setw(22)
            << result.consumer_throughput

            << '\n';
    }


    return 0;
}

Writing distributed-profiler/benchmarks/queue_capacity_benchmark.cpp


In [14]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/benchmarks/queue_capacity_benchmark.cpp \
    -pthread \
    -o queue_capacity_benchmark

In [15]:
!./queue_capacity_benchmark


=== Queue Capacity Experiment ===

Capacity  Blocks      Block Ratio   Block Time    Avg Block     Max Queue   Queue Util    Runtime     
------------------------------------------------------------------------------------------------------
1         99          99.00%        0.999509      10.0961       1           100.00%       1.02075     
8         91          91.00%        0.923681      10.1503       8           100.00%       1.01592     
16        83          83.00%        0.846137      10.1944       16          100.00%       1.01956     
32        67          67.00%        0.680501      10.1567       32          100.00%       1.0161      
64        35          35.00%        0.355952      10.1701       64          100.00%       1.01643     
128       0           0.000%        0             0             99          77.343%       1.01627     

Throughput comparison

Capacity    Producer events/s     Consumer events/s     
--------------------------------------------------------
1 